# 第6周 · 无监督学习、梯度与文本基线

**本周结束时你能做到**：用 PCA 把 64 维的手写数字压到 2 维并解释"解释方差"；用 k-means 聚类并用 ARI 打分；对任何函数亲手求导并用数值差分验证（第7周 micrograd 的直接前置）；为 IMDB 影评情感分类搭起一座**基线阶梯**——它是之后 TextCNN、Transformer、DistilBERT 的固定参照。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week06 as ck

---
## S1 · 没有标签怎么办：PCA 与 k-means

前几周的模型都有"答案"（标签）可学。**无监督学习**在没有标签时找结构：

- **k-means 聚类**：把样本分成 k 群，每群围着自己的中心。
- **PCA（主成分分析）**：找数据方差最大的几个方向，把高维数据投影上去——压缩与可视化的标准工具。

先在一份能看见的二维合成数据上体会：

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

Xb, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xb)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(Xb[:, 0], Xb[:, 1], s=12, color="gray")
axes[0].set_title("What the algorithm sees: no labels")
axes[1].scatter(Xb[:, 0], Xb[:, 1], c=km4.labels_, s=12, cmap="tab10")
axes[1].scatter(km4.cluster_centers_[:, 0], km4.cluster_centers_[:, 1],
                marker="x", s=120, color="k")
axes[1].set_title("KMeans recovers the 4 clusters (x = centres)")
for a in axes:
    a.set_xlabel("x1")
    a.set_ylabel("x2")
plt.tight_layout()
plt.show()

### digits 的 PCA：64 维里到底有多少"真信息"？

第2周见过的手写数字：1797 张图、每张展平成 64 维。PCA 的 `explained_variance_ratio_` 告诉我们每个主成分方向"解释"了总方差的百分之几。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
Xd, yd = digits.data, digits.target

# ✍️ 跟着老师一起写
pass
print("前两个主成分的解释方差比:", np.round(evr[:2], 3))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.arange(1, 65), np.cumsum(evr), ".-")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("About 20 of 64 components keep ~90% of the variance")
plt.show()
ck.pca_evr(evr[:2])

In [ ]:
Z = PCA(n_components=2).fit_transform(Xd)
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=yd, cmap="tab10", s=8)
fig.colorbar(sc, label="True digit")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Two components already separate several digits")
plt.show()

只用 2/64 的维度，一些数字已经各占一角——而我们**全程没有用标签**（上色只是事后对答案）。

### k-means 聚 10 类，用 ARI 对答案

digits 恰好有 10 类，让 k-means 盲聚成 10 群，再和真实标签比。直接比准确率不行——聚类的"3 号簇"不一定对应数字 3。**ARI（调整兰德指数）** 解决了这一点：它看的是每一对样本："这两个在不在同一群"这件事，聚类结果和真实标签一致的程度（该在一起的在一起、该分开的分开），并扣掉了随机分群也能碰巧对上的部分——随机聚类约得 0，完美得 1。

**注意**：这里直接用原始像素（0–16），**不做标准化**——像素本来就同一量纲。

In [ ]:
from sklearn.metrics import adjusted_rand_score

# ✍️ 跟着老师一起写
pass
print("ARI =", round(ari, 3))
ck.kmeans_ari(ari)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(km10.cluster_centers_[i].reshape(8, 8), cmap="gray_r")
    ax.axis("off")
fig.suptitle("Cluster centres look like digits -- found without any labels")
plt.show()

> ⚠️ **一句话警示**：t-SNE 和 UMAP 的图只能用来"看一眼"——它们为了好看会扭曲几何，图上的**点间距离、簇的大小与间距都不可解读**，更不要在上面量距离做下游结论。

---
## S2 · 数学工作坊 2：导数、梯度与链式法则

第4周我们直接用了"MSE 对 w 的梯度"这个公式；第7周我们要从零搭出自动求导。今天把中间的数学补上。**导数**回答一个问题：输入动一点点，输出动多少？

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

右边不取极限、让 $h$ 取个很小的数（如 $10^{-5}$），就是**数值差分**。实际验证时我们用更准的**中心差分** $\frac{f(x+h) - f(x-h)}{2h}$：两侧各走一小步，误差比单侧小得多——本节课每个手推结果都要用它验证。（小提醒：$|x|$、ReLU 这类函数在"尖角"处导数不存在，中心差分却照样会算出一个数，比如 ReLU 在 0 处得 0.5——那不是导数。）先把它写成函数：

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    """数值差分估计 f 在 x 处的导数"""
    # ✍️ 跟着老师一起写
    pass

print(numerical_gradient(lambda x: x ** 2, 3.0))   # (x^2)' = 2x，在 x=3 处应约等于 6
ck.numerical_gradient(numerical_gradient)

### 复合函数与计算图

$f(x) = (3x + 2)^2$ 是两步运算的**复合**。把它画成计算图：

```
x ──(×3，+2)──▶ u ──(平方)──▶ f
```

**链式法则**：复合函数的导数 = 每一段导数的乘积。

$$\frac{df}{dx} = \frac{df}{du} \cdot \frac{du}{dx} = 2u \cdot 3$$

从输出端往输入端一段一段地乘回去——第7周的反向传播就是把这件事自动化。

In [ ]:
x = 1.0
u = 3 * x + 2        # 前向第一步：u = 5
f = u ** 2           # 前向第二步：f = 25

# ✍️ 跟着老师一起写
pass
print("链式法则手推:", df_dx)
print("数值差分验证:", numerical_gradient(lambda x: (3 * x + 2) ** 2, x))

### 偏导数与梯度向量

函数有多个输入时，**偏导数**是"只动这一个输入、冻住其他输入"的导数；把所有偏导数排成的向量叫**梯度**，它指向函数上升最快的方向（所以梯度下降往反方向走）。

对 $f(w_1, w_2) = w_1^2 + 3 w_2$：$\partial f / \partial w_1 = 2 w_1$，$\partial f / \partial w_2 = 3$。

In [ ]:
def f2(w1, w2):
    return w1 ** 2 + 3 * w2

w1, w2 = 2.0, -1.0
# ✍️ 跟着老师一起写
pass
numeric = np.array([numerical_gradient(lambda a: f2(a, w2), w1),
                    numerical_gradient(lambda b: f2(w1, b), w2)])
print("手推梯度:", grad)
print("数值验证:", numeric)

### 手推 MSE 对 w 的梯度（与第4周对上）

单参数线性模型 $\hat{y}_i = w x_i$，损失 $L(w) = \frac{1}{n}\sum_i (w x_i - y_i)^2$。

对每一项用链式法则：外层是平方（导数 $2(w x_i - y_i)$），内层 $w x_i - y_i$ 对 $w$ 的导数是 $x_i$。所以

$$\frac{dL}{dw} = \frac{1}{n}\sum_i 2 (w x_i - y_i) \, x_i$$

——这正是第4周 `gd_step` 里 `grad_w` 那一行（`2 / n * X.T @ (y_hat - y)`，只有一个特征时就是上式；第4周 S2 手算的那一步也是它）。验证：

In [ ]:
rng = np.random.default_rng(0)
x_data = rng.uniform(-1, 1, 50)
y_data = 2.5 * x_data + rng.normal(0, 0.1, 50)


def mse(w):
    return np.mean((w * x_data - y_data) ** 2)


w = 1.0
# ✍️ 跟着老师一起写
pass
print("手推梯度:", round(grad_w, 6))
print("数值差分:", round(numerical_gradient(mse, w), 6))

两个数字对上了。第7周的 micrograd 做的事：把"前向逐步计算、反向逐段乘导数"写成通用程序，让**任何**计算图都能自动得到梯度。

---
## S3 · 文本主线启动：IMDB 影评情感分类

从今天起到第14周，我们反复使用同一份数据：IMDB 影评情感数据集（Maas et al. 2011）的**课程固定子集**——训练 5000 条、测试 2000 条，列 `text` 与 `label`（0 = 差评，1 = 好评），两类均衡、已去重并打乱。

In [ ]:
train = data.load_imdb("train")
test = data.load_imdb("test")
print(train.shape, test.shape)
print(train["label"].value_counts().to_dict())
print("\n一条好评的开头：\n", train.loc[train["label"] == 1, "text"].iloc[0][:300], "...")

### 把文字变成数字：词袋与 TF-IDF

模型只吃数字。最朴素的编码是**词袋**（bag of words）：数每个词出现了几次，语序全部丢掉。先在玩具语料上看 `CountVectorizer` 做了什么：

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

toy = ["a great great movie", "a terrible movie"]
v = CountVectorizer()
M = v.fit_transform(toy)
print(v.get_feature_names_out())
print(M.toarray())

**TF-IDF** 在词频基础上，把"哪篇都出现"的常见词（the、movie）压低权重，突出有区分度的词。两者都用 `min_df=2` 丢掉只在 1 条影评里出现的词（多是拼写错误和人名）。

### 基线阶梯

研究的第一铁律：**任何花哨模型，都必须先打败它脚下的阶梯**。今天搭的这座阶梯，就是第10周 TextCNN、第11周 Transformer、第14周 DistilBERT 的固定参照。

- 第 0 级：多数类基线（闭着眼睛猜**训练集**里最多的那类）
- 第 1 级：词袋 + 逻辑回归
- 第 2 级：TF-IDF + 逻辑回归
- 对照组：TF-IDF + 随机森林

In [ ]:
from sklearn.dummy import DummyClassifier

# 多数类基线：预测"训练集"里最多的那一类，再到测试集上打分——测试集不参与任何决定。
# （两类均衡时它和"测试集多数类的占比"相同；类别不均衡时两种写法会不一样，前者才不偷看测试集。）
dummy = DummyClassifier(strategy="most_frequent").fit(train["text"], train["label"])
majority_acc = dummy.score(test["text"], test["label"])
print("第 0 级 · 多数类基线:", majority_acc)

In [ ]:
from sklearn.linear_model import LogisticRegression

# ✍️ 跟着老师一起写
pass
print(f"第 1 级 · 词袋 + 逻辑回归: {bow_acc:.4f}   （词表大小 {X_bow.shape[1]}）")

In [ ]:
# ✍️ 跟着老师一起写
pass
print(f"第 2 级 · TF-IDF + 逻辑回归: {tfidf_acc:.4f}")
ck.baseline_ladder(majority_acc, bow_acc, tfidf_acc)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 与教案一致限制到 2 万维。这个子集 min_df=2 的词表本来只有约 2.2 万维，限制只砍掉约 9% 的列，
# 速度和准确率都几乎不变；完整 IMDB（2.5 万条训练）的词表约 7.5 万维，那时限制维度才有实际意义。
tfidf20k = TfidfVectorizer(min_df=2, max_features=20000)
X_20k = tfidf20k.fit_transform(train["text"])
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_20k, train["label"])
rf_acc = rf.score(tfidf20k.transform(test["text"]), test["label"])
print(f"对照组 · TF-IDF + 随机森林: {rf_acc:.4f}")

In [ ]:
names = ["Majority", "BoW + LR", "TF-IDF + LR", "TF-IDF + RF"]
vals = [majority_acc, bow_acc, tfidf_acc, rf_acc]
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(names, vals, color=["gray", "C0", "C0", "C1"])
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("Test accuracy")
ax.set_title("Simple baselines are strong: TF-IDF + LR reaches ~0.87")
plt.show()

记住这几个数字。之后任何"深度学习模型"如果没有明显越过 0.87 这一级，它就还没证明自己值那些算力。

顺便看看逻辑回归学到了什么——权重最大/最小的词：

In [ ]:
words = np.array(tfidf.get_feature_names_out())
w = clf_tfidf.coef_[0]
print("最'好评'的词:", words[np.argsort(w)[-10:][::-1]])
print("最'差评'的词:", words[np.argsort(w)[:10]])

---
## S4 · 研究方法微课：最小实验单元

从下周开始，你做的每一个对比都要长成下面这个样子——**最小实验单元**：

> **一个基线 + 一次受控修改 + 至少 3 个随机种子 + 均值 ± 标准差**

- **一次只改一个东西**（自变量），其余全部冻住（控制变量）；
- **至少 3 个种子**：单次运行的差异可能纯属运气，没有标准差的对比无法解读；
- 结论只能是三种之一：显著更好 / 显著更差 / **差异小于噪声**——第三种也是合法发现。

每做一个单元，就写一条**实验记录**（格式见 `templates/lab_notebook.md`）。现场示范——把今天 S3 的"词袋 → TF-IDF"写成一条：

---

## 实验记录 #0 · 日期：2026-10-30（课堂示范）

- 研究问题：在 IMDB 课程子集上，TF-IDF 编码是否优于原始词频（词袋）？
- 假设：TF-IDF 压低常见词权重，测试准确率更高；若两者差距 < 1 个百分点则视为无实质差异。
- 自变量：文本编码方式（CountVectorizer → TfidfVectorizer）。
- 控制变量：同一训练/测试划分（课程固定子集）、`min_df=2`、`LogisticRegression(max_iter=1000)`。
- 基线：词袋 + 逻辑回归 = 0.8550。
- 数据与划分方式：课程固定子集 5000/2000，向量化器只在训练集上 fit（防泄漏）。
- 随机种子：无随机成分（划分固定、LBFGS 确定性）——**严格说本条只跑了"1 个种子"，不合最小单元标准**，作业里你将补上多种子版本。
- 结果：TF-IDF = 0.8665（+1.2 个百分点）。
- 解释：常见词权重被压低后，有区分度的情感词相对更突出。
- 局限：单一划分、单一超参数设定；提升幅度接近"1 个百分点"的判断线。
- 下一步：以 n-gram 范围为自变量、3 个训练子集为"种子"，做一次合格的最小实验单元（本周作业 A）。注意这样的"种子"只反映"换一批训练数据"的波动，不含测试集本身的抽样误差，会低估不确定性（作业 A 有详细说明）。

---

本周课堂练习是一次**泄漏排查演练**（打开 `06_exercises.ipynb`）；作业 A 是你的**第一条正式实验记录**，之后所有研究作业都按它的格式评分。

### 本周小结

- PCA：找方差最大的方向；`explained_variance_ratio_` 告诉你留了多少信息
- k-means：无标签聚类；与真实标签比较用 ARI；**预处理会改变结论**（digits 上标准化反而更差）
- t-SNE/UMAP 的图上距离不可解读
- 导数 = 输入动一点点输出动多少；链式法则 = 逐段导数相乘；一切手推都用数值差分验证
- 基线阶梯：多数类 0.500 → 词袋+LR 0.855 → TF-IDF+LR 0.867——深度模型的固定参照
- 最小实验单元：基线 + 一次受控修改 + ≥3 种子 + 均值±标准差，并写实验记录